# EDA - Heart Disease Dataset

Exploring the raw dataset (data/heart.csv - the real UCI/Kaggle multi-site
heart disease data) before preprocessing/modeling: shape, missing values,
distributions, correlations, etc.

Note: the raw file uses `num` (a multi-class 0-4 label) rather than a
binary `target`, and several columns (sex, cp, restecg, exang, slope,
thal) are text categories rather than numeric codes - this notebook
derives `target` the same way src/preprocessing.py does, so the two stay
consistent.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
%matplotlib inline

In [ ]:
df = pd.read_csv('../data/heart.csv', na_values='?')

# Derive the binary target from the original multi-class "num" label,
# the same way src/preprocessing.py does for training.
df['target'] = (df['num'] > 0).astype(int)

df.head()


### basic info about the dataset

In [ ]:
print("shape:", df.shape)
df.info()

In [ ]:
df.describe()

### checking missing values

In [ ]:
df.isnull().sum()

In [ ]:
# several columns have missing values - ca, thal, and slope are the
# least complete, chol and thalch have relatively few missing values.
# All are handled via median/mode imputation during preprocessing.
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_percent[missing_percent > 0].sort_values(ascending=False)


### target column balance

In [ ]:
df['target'].value_counts()

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x='target', data=df)
plt.title('Disease vs No Disease count')
plt.xlabel('target (0 = no disease, 1 = disease)')
plt.show()

# looks fairly balanced so we dont need to worry about class imbalance too much

### distribution of numeric features

In [ ]:
num_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']

df[num_cols].hist(figsize=(12,8), bins=20)
plt.tight_layout()
plt.show()


### age distribution wrt target

In [ ]:
plt.figure(figsize=(7,5))
sns.histplot(data=df, x='age', hue='target', bins=20, kde=True)
plt.title('Age distribution by target')
plt.show()

### correlation heatmap

In [ ]:
plt.figure(figsize=(10,8))
corr = df.select_dtypes(include='number').drop(columns=['id']).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap (numeric columns only)')
plt.show()


In [ ]:
# checking which features correlate most with target
corr['target'].sort_values(ascending=False)

### boxplots to check outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))

sns.boxplot(y=df['chol'], ax=axes[0])
axes[0].set_title('cholesterol')

sns.boxplot(y=df['trestbps'], ax=axes[1])
axes[1].set_title('resting bp')

sns.boxplot(y=df['oldpeak'], ax=axes[2])
axes[2].set_title('oldpeak')

plt.tight_layout()
plt.show()

### chest pain type vs target

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='cp', hue='target', data=df)
plt.title('Chest Pain Type vs Target')
plt.show()

### observations

- `ca` (611 missing), `thal` (486 missing), and `slope` (309 missing) are
  the least complete columns; `chol` and `thalch` have relatively few
  missing values. All are handled via median/mode imputation in
  src/preprocessing.py.
- The target is mildly imbalanced (509 disease / 411 no-disease), so
  accuracy is still a reasonable headline metric, though ROC-AUC/F1 give a
  fuller picture given the imbalance.
- `ca`, `oldpeak`, and `age` show the strongest positive correlation with
  disease risk; `thalch` (max heart rate) shows the strongest negative
  correlation.
- `id` and `dataset` are excluded from this analysis (and from the trained
  model - see src/preprocessing.py): `id` is just a row identifier, and
  `dataset` records which clinical site collected the record, not a
  patient attribute - including it would leak site information into the
  model rather than teaching real clinical patterns.
